# **Definición del problema y dataset INMET**

**Objetivo:** Definir formalmente el problema de forecasting meteorológico horario como un problema de **series de tiempo**, describir el dataset INMET y justificar las decisiones metodológicas.

## **1. Naturaleza del problema: Series de Tiempo vs. Regresión**

Este trabajo aborda un problema de **forecasting de series de tiempo multivariadas** (time series forecasting), fundamentalmente distinto a un problema de regresión estándar. Las diferencias clave son:

| Característica | Regresión estándar | Series de tiempo (este trabajo) |
|---|---|---|
| Observaciones | i.i.d. | Dependientes temporalmente |
| Estructura de datos | Sin orden | Orden temporal importa |
| Validación | K-Fold aleatorio | Partición temporal estricta |
| Objetivo | Predecir $y$ dado $\mathbf{x}$ | Predecir $y_{t+1:t+H}$ dado el historial |
| Riesgo principal | Sobreajuste | Data leakage temporal |

La **dependencia temporal** entre observaciones (autocorrelación) viola el supuesto de independencia de la regresión clásica. Un modelo que use información futura para predecir el pasado produciría resultados artificialmente optimistas — esto se conoce como *data leakage temporal* y es el error metodológico más común en este dominio.

### ¿Por qué no se usó timeseries-cv?

La librería `timeseries-cv` implementa validación cruzada temporal con múltiples folds. En este trabajo se optó por una **partición temporal fija** (train/val/test) por las siguientes razones:

1. **Volumen de datos:** Con 2.63M filas y 38 estaciones, el entrenamiento de 7 modelos × 2 seeds ya demanda ~200 horas de GPU. Multiplicar por K folds haría el experimento computacionalmente inviable.
2. **Horizonte largo:** El horizonte de predicción es de hasta 168 horas (7 días). Con folds demasiado pequeños, el modelo no tendría suficiente contexto histórico.
3. **Partición cronológica estricta:** Se garantiza que el conjunto de test siempre es temporalmente posterior al de entrenamiento, evitando data leakage. Esto es metodológicamente equivalente a un único fold de walk-forward validation.

## **2. Definición formal del problema**

Sea $y_t \in \mathbb{R}$ la temperatura horaria en una estación meteorológica y
$\mathbf{x}_t \in \mathbb{R}^F$ las $F=6$ covariables meteorológicas observadas (precipitación, humedad relativa, presión, radiación, velocidad y dirección del viento). 

Buscamos un modelo $f_\theta$ tal que:
$$
\hat{y}_{t+1:t+H} = f_\theta\bigl(\mathbf{x}_{t-L+1:t},\, y_{t-L+1:t}\bigr)
$$

donde:
- $L = 168$ horas es la **ventana de lookback** (contexto histórico de 7 días)
- $H \in \{24, 72, 168\}$ horas son los **horizontes de predicción** evaluados (1, 3 y 7 días)
- El objetivo es minimizar $\mathrm{RMSE}$ en el conjunto de test temporal, con MAE, R² y sMAPE como métricas complementarias

### Partición temporal

Para evitar data leakage, la partición es **estrictamente cronológica**:

| Partición | Período | Uso |
|---|---|---|
| Train | 2000–2021 | Ajuste de parámetros |
| Validación | 2022 | Early stopping y selección de hiperparámetros |
| Test | 2023 | Evaluación final (nunca vista durante entrenamiento) |

Esta partición garantiza que el modelo nunca tiene acceso a datos futuros durante el entrenamiento.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils import load_yaml, set_seed
cfg = load_yaml(PROJECT_ROOT / "config" / "config.yaml")
set_seed(cfg['project']['seed'])
cfg['task']

{'target': 'temp_c',
 'multi_target': [],
 'exog': ['humidity_pct', 'pressure_mb', 'radiation_kj_m2', 'wind_speed_ms'],
 'freq': 'h',
 'lookback': 168,
 'horizon': 168}

## **2. Dataset INMET**



In [2]:
from src.data.ingest_inmet import ingest


In [3]:
from src.data.clean import clean_station

interim_dir = PROJECT_ROOT / cfg['paths']['data_interim']
station = next(
    (p for p in sorted(interim_dir.iterdir()) if p.is_dir() and any(p.glob("*.csv"))),
    None,
)
if station is not None:
    df = clean_station(station, cfg)
    df.head()
else:
    print(f"Sin carpetas de estacion con CSV en {interim_dir}")

[2026-05-22 08:51:59] INFO    src.data.clean :: Estación A001 — 61368 filas, 6.3% NaN tras limpieza
